# Computational performance analysis - Logistic Regression

Global imports

In [ ]:
from river import linear_model, optim, evaluate, metrics
from tabulate import tabulate
import sys
import importlib.util
import os
import time
import tracemalloc

Initial environment configuration

In [ ]:
# Ensure custom capymoa wrapper is loaded over the installed one
wrapper_path = os.path.abspath(os.path.join(os.getcwd(), "..", "capymoa", "src"))
if wrapper_path not in sys.path:
    sys.path.insert(0, wrapper_path)

os.environ["CAPYMOA_MOA_JAR"] = os.path.abspath(os.path.join(os.getcwd(), "..", "custom_moa_full.jar"))

Global variables

In [ ]:
DEFAULT_LR = 0.01
DEFAULT_BIAS_LR = 0.01
DEFAULT_L1 = 0.0
DEFAULT_L2 = 0.0
DEFAULT_CLIP = 1e12
DEFAULT_BIAS_INIT = 0.0
MAX_INSTANCES = 100000
SEED = 42

Dynamically load the custom LogisticRegression over the installed capymoa package

In [ ]:
file_path = os.path.abspath(os.path.join(os.getcwd(), "..", "capymoa", "src", "capymoa", "classifier", "_logistic_regression.py"))
spec = importlib.util.spec_from_file_location("capymoa.classifier._logistic_regression", file_path)
module = importlib.util.module_from_spec(spec)
sys.modules["capymoa.classifier._logistic_regression"] = module
spec.loader.exec_module(module)
LogisticRegression = module.LogisticRegression

Global functions

In [ ]:
def adaptStreamForRiver(stream):
    data = []

    for i, instance in enumerate(stream):
        if(i > MAX_INSTANCES): break

        # features
        x = {f"f{j}": float(v) for j, v in enumerate(instance.x)}

        # label
        y = instance.y_index
        
        data.append((x, y))
    
    return data

In [ ]:
from capymoa.evaluation import prequential_evaluation # We need to import capymoa's methods after setting the custom jar

def evaluateStream(stream_factory, lr=DEFAULT_LR, b_lr=DEFAULT_BIAS_LR, l1=DEFAULT_L1, l2=DEFAULT_L2, clip=DEFAULT_CLIP, bias_init=DEFAULT_BIAS_INIT):
    # stream_factory must be deterministic. It is the constructor of the stream.

    capyMoaResults = _evaluateStreamOnCapyMoa(stream_factory(), lr, b_lr, l1, l2, clip, bias_init)

    riverStream = adaptStreamForRiver(stream_factory())

    riverResults = _evaluateStreamOnRiver(riverStream, lr, b_lr, l1, l2, clip, bias_init)

    table = []
    for key in ["Time (s)", "Memory (MB)"]:
        capy = capyMoaResults[key]
        river = riverResults[key]

        table.append([
            key,
            f"{capy:.4f}",
            f"{river:.4f}",
            f"{(capy - river):+.4f}"
        ])

    for key in ["Accuracy", "F1"]:
        capy = capyMoaResults[key]
        river = riverResults[key]

        table.append([
            key,
            f"{capy:.2f}%",
            f"{river:.2f}%",
            f"{(capy - river):+.2f}%"
        ])

    print("\n--- Comparison CapyMOA vs River ---\n")
    print(
        tabulate(
            table,
            headers=["Metric", "CapyMOA", "River", "Delta"],
            tablefmt="fancy_grid"
        )
    )

def _evaluateStreamOnCapyMoa(stream, lr, b_lr, l1, l2, clip, bias_init):
    log_reg_capymoa = LogisticRegression(
        schema=stream.get_schema(),
        learning_rate=lr,
        bias_learning_rate=b_lr,
        l1_penalty=l1,
        l2_penalty=l2,
        clip_gradient=clip,
        bias_init=bias_init
    )

    tracemalloc.start()
    start_time = time.time()

    # prequential evaluation using CapyMOA built-in function
    results = prequential_evaluation(
        stream=stream,
        learner=log_reg_capymoa,
        max_instances=MAX_INSTANCES
    )

    end_time = time.time()
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    metrics = {
        "Accuracy": results['cumulative'].accuracy(),
        "F1": results['cumulative'].f1_score(),
        "Time (s)": end_time - start_time,
        "Memory (MB)": peak_memory / (1024 * 1024)
    }

    return metrics

def _evaluateStreamOnRiver(stream, lr, b_lr, l1, l2, clip, bias_init):
    log_reg_river = linear_model.LogisticRegression(
        optimizer=optim.SGD(lr),
        intercept_lr=b_lr,
        l1=l1,
        l2=l2,
        clip_gradient=clip,
        intercept_init=bias_init
    )

    metric = (
        metrics.Accuracy() +
        metrics.F1()
    )

    tracemalloc.start()
    start_time = time.time()

    # prequential evaluation using River built-in function
    result = evaluate.progressive_val_score(
        dataset=stream,
        model=log_reg_river,
        metric=metric
    )

    end_time = time.time()
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    results = {}
    results["Accuracy"] = float(result[0].get()*100)
    results["F1"] = float(result[1].get()*100)

    results["Time (s)"] = end_time - start_time
    results["Memory (MB)"] = peak_memory / (1024 * 1024)

    return results

## Electricity dataset

In [ ]:
from capymoa.datasets import Electricity

evaluateStream(Electricity)

## ElectricityTiny dataset

In [ ]:
from capymoa.datasets import ElectricityTiny

evaluateStream(ElectricityTiny)

## RandomRBFGenerator

In [ ]:
from capymoa.stream.generator import RandomRBFGenerator

def make_stream():
    return RandomRBFGenerator(
        number_of_classes=2,
        number_of_attributes=50,
        number_of_centroids=100,
        model_random_seed=SEED
    )

evaluateStream(make_stream)

## Hyper100k dataset

In [ ]:
from capymoa.datasets import Hyper100k

evaluateStream(Hyper100k)

## SEA dataset generator

In [ ]:
from capymoa.stream.generator import SEA

def make_stream():
    return SEA(
        instance_random_seed=SEED,
        function=1,
        balance_classes=False,
        noise_percentage=10,
    )

evaluateStream(make_stream)

## HyperPlaneClassification dataset

In [ ]:
from capymoa.stream.generator import HyperPlaneClassification

def make_stream():
    return HyperPlaneClassification(
        instance_random_seed=SEED,
        number_of_classes=2,
        number_of_attributes=10,
        number_of_drifting_attributes=2,
        magnitude_of_change=0.0,
        noise_percentage=5,
        sigma_percentage=10,
    )

evaluateStream(make_stream)

## RandomTreeGenerator

In [ ]:
from capymoa.stream.generator import RandomTreeGenerator

def make_stream():
    return RandomTreeGenerator(
        instance_random_seed=SEED,
        tree_random_seed=SEED,
        num_classes=2,
        num_nominals=0,
        num_numerics=5,
        max_tree_depth=5,
        first_leaf_level=3,
        leaf_fraction=0.15,
    )

evaluateStream(make_stream)

## Electricity dataset (changed model parameters)

### L2

In [ ]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, l2=0.01)

### L1

In [ ]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, l1=0.01)

### Learning rate

In [ ]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, lr=0.1, b_lr=0.1)

### Gradient clipping

In [ ]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, clip=1)

### Bias initialization

In [ ]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, bias_init=3)